# RuleShift Benchmark v1 Kaggle Staging Dry Run

This notebook validates the packaged frozen artifact bundle and runs a staging dry run over the implemented v1 benchmark flow. It keeps Binary as the sole headline path, includes Narrative as the required same-episode robustness companion, and uses a deterministic notebook-local adapter only to verify parsing, scoring, and report rendering from packaged artifacts.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
import json
from pathlib import Path
import sys


def _is_repo_root(candidate: Path) -> bool:
    return (
        (candidate / "src").is_dir()
        and (candidate / "packaging" / "kaggle" / "frozen_artifacts_manifest.json").is_file()
    )


def _candidate_repo_roots() -> tuple[Path, ...]:
    candidates: list[Path] = []
    seen: set[Path] = set()

    for origin in (Path.cwd().resolve(),):
        for candidate in (origin, *origin.parents):
            if candidate not in seen:
                seen.add(candidate)
                candidates.append(candidate)

    for search_root in (Path("/kaggle/input"), Path("/kaggle/working")):
        if not search_root.exists():
            continue
        for manifest_path in search_root.rglob("frozen_artifacts_manifest.json"):
            candidate = manifest_path.parents[2]
            if candidate not in seen:
                seen.add(candidate)
                candidates.append(candidate)

    return tuple(candidate for candidate in candidates if _is_repo_root(candidate))


REPO_ROOT = next(iter(_candidate_repo_roots()), None)
if REPO_ROOT is None:
    raise FileNotFoundError(
        "Could not resolve the packaged repo root from the notebook environment. "
        "Expected a bundled src/ directory and packaging/kaggle/frozen_artifacts_manifest.json."
    )

if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from core.kaggle import load_kaggle_staging_manifest, validate_kaggle_staging_manifest
from core.metrics import compute_metrics
from core.model_execution import ModelAdapter, ModelMode, ModelRawResult, ModelRequest, ModelRunConfig
from core.model_runner import run_model_benchmark
from core.private_split import load_private_split, resolve_private_dataset_root
from core.splits import load_frozen_split, load_split_manifest
from tasks.ruleshift_benchmark.render import render_binary_prompt, render_narrative_prompt

try:
    PRIVATE_DATASET_ROOT = resolve_private_dataset_root()
except FileNotFoundError:
    PRIVATE_DATASET_ROOT = None

validate_kaggle_staging_manifest(REPO_ROOT)
manifest = load_kaggle_staging_manifest(REPO_ROOT)

{
    "repo_root": str(REPO_ROOT),
    "bundle_version": manifest["bundle_version"],
    "task_id": manifest["task_id"],
    "benchmark_versions": manifest["benchmark_versions"],
    "current_emitted_difficulty_labels": manifest["current_emitted_difficulty_labels"],
    "reserved_difficulty_labels": manifest["reserved_difficulty_labels"],
}


In [ ]:
card_path = REPO_ROOT / "packaging" / "kaggle" / "BENCHMARK_CARD.md"
readme_path = REPO_ROOT / "README.md"
runbook_path = REPO_ROOT / "packaging" / "kaggle" / "PRIVATE_SPLIT_RUNBOOK.md"
r13_path = REPO_ROOT / "tests" / "fixtures" / "release_r13_validity_report.json"
r15_path = REPO_ROOT / "tests" / "fixtures" / "release_r15_reaudit_report.json"

card_excerpt = "\n".join(card_path.read_text(encoding="utf-8").splitlines()[:20])
r13_report = json.loads(r13_path.read_text(encoding="utf-8"))
r15_report = json.loads(r15_path.read_text(encoding="utf-8"))

{
    "benchmark_card": str(card_path.relative_to(REPO_ROOT)),
    "readme": str(readme_path.relative_to(REPO_ROOT)),
    "private_split_runbook": str(runbook_path.relative_to(REPO_ROOT)),
    "benchmark_card_excerpt": card_excerpt,
    "r13_gate_passed": r13_report["passed"],
    "r13_summary": r13_report["comparison_summary"],
    "r15_audit_note": r15_report["audit_note"],
    "source_of_truth": manifest["source_of_truth"],
}


In [ ]:
_PUBLIC_PARTITIONS = ("dev", "public_leaderboard")

split_records = {
    partition: load_frozen_split(partition)
    for partition in _PUBLIC_PARTITIONS
}

if PRIVATE_DATASET_ROOT is not None:
    split_records["private_leaderboard"] = load_private_split(PRIVATE_DATASET_ROOT)
    print("Private evaluation dataset: attached")
else:
    print("Private evaluation dataset: not attached (private eval will be skipped)")

_LOADED_PARTITIONS = tuple(split_records)

split_summary = {
    partition: {
        "episode_split": split_records[partition][0].episode.split.value if split_records[partition] else "unknown",
        "episode_count": len(split_records[partition]),
    }
    for partition in _LOADED_PARTITIONS
}
split_summary


In [ ]:
def _labels_text(labels) -> str:
    return ", ".join(label.value for label in labels)


def _narrative_text(labels) -> str:
    return "\n".join(
        (
            "rule_before: staged pre-shift rule",
            "shift_evidence: staged later evidence revised the rule",
            "rule_after: staged post-shift rule",
            f"final_decision: {_labels_text(labels)}",
        )
    )


def _flip_first_label(labels):
    first_label = labels[0]
    flipped_first_label = type(first_label)("repel" if first_label.value == "attract" else "attract")
    return (flipped_first_label, *labels[1:])


@dataclass
class StagingDryRunAdapter:
    responses: dict[tuple[ModelMode, str], ModelRawResult]

    def generate(self, request: ModelRequest, config: ModelRunConfig) -> ModelRawResult:
        del config
        return self.responses[(request.mode, request.prompt_text)]


responses: dict[tuple[ModelMode, str], ModelRawResult] = {}
for split_name in _LOADED_PARTITIONS:
    episodes = tuple(record.episode for record in split_records[split_name])
    for index, episode in enumerate(episodes):
        binary_prompt = render_binary_prompt(episode)
        binary_request = ModelRequest(
            provider_name="staging-dry-run",
            model_name="packaged-frozen-artifacts",
            prompt_text=binary_prompt,
            mode=ModelMode.BINARY,
        )
        responses[(ModelMode.BINARY, binary_prompt)] = ModelRawResult.from_request(
            binary_request,
            response_text=_labels_text(episode.probe_targets),
        )

        narrative_prompt = render_narrative_prompt(episode)
        narrative_request = ModelRequest(
            provider_name="staging-dry-run",
            model_name="packaged-frozen-artifacts",
            prompt_text=narrative_prompt,
            mode=ModelMode.NARRATIVE,
        )
        if split_name == "dev" and index == 0:
            narrative_text = "Reasoning without a valid final line."
        elif split_name == "public_leaderboard" and index == 0:
            narrative_text = _narrative_text(_flip_first_label(episode.probe_targets))
        else:
            narrative_text = _narrative_text(episode.probe_targets)
        responses[(ModelMode.NARRATIVE, narrative_prompt)] = ModelRawResult.from_request(
            narrative_request,
            response_text=narrative_text,
        )

adapter = StagingDryRunAdapter(responses)
benchmark_results_by_split = {
    split_name: run_model_benchmark(
        tuple(record.episode for record in split_records[split_name]),
        adapter,
        provider_name="staging-dry-run",
        model_name="packaged-frozen-artifacts",
        modes=(ModelMode.BINARY, ModelMode.NARRATIVE),
    )
    for split_name in _LOADED_PARTITIONS
}

overall_metrics = compute_metrics(
    binary_predictions=tuple(
        row.parsed_prediction
        for split_name in _LOADED_PARTITIONS
        for row in benchmark_results_by_split[split_name].mode_results[0].rows
    ),
    binary_targets=tuple(
        row.target
        for split_name in _LOADED_PARTITIONS
        for row in benchmark_results_by_split[split_name].mode_results[0].rows
    ),
    narrative_predictions=tuple(
        row.parsed_prediction
        for split_name in _LOADED_PARTITIONS
        for row in benchmark_results_by_split[split_name].mode_results[1].rows
    ),
    narrative_targets=tuple(
        row.target
        for split_name in _LOADED_PARTITIONS
        for row in benchmark_results_by_split[split_name].mode_results[1].rows
    ),
)

pairing_checks = {}
for split_name in _LOADED_PARTITIONS:
    episodes = tuple(record.episode for record in split_records[split_name])
    binary_rows, narrative_rows = benchmark_results_by_split[split_name].mode_results
    pairing_checks[split_name] = {
        "same_episode_order": [row.episode_id for row in binary_rows.rows] == [episode.episode_id for episode in episodes]
        and [row.episode_id for row in narrative_rows.rows] == [episode.episode_id for episode in episodes],
        "same_probe_targets": [tuple(row.target) for row in binary_rows.rows] == [tuple(episode.probe_targets) for episode in episodes]
        and [tuple(row.target) for row in narrative_rows.rows] == [tuple(episode.probe_targets) for episode in episodes],
    }

dry_run_summary = {
    "provider_name": "staging-dry-run",
    "model_name": "packaged-frozen-artifacts",
    "prompt_modes": [ModelMode.BINARY.value, ModelMode.NARRATIVE.value],
    "overall_metrics": {
        "post_shift_probe_accuracy": overall_metrics.post_shift_probe_accuracy,
        "binary_accuracy": overall_metrics.binary_accuracy,
        "narrative_accuracy": overall_metrics.narrative_accuracy,
        "parse_valid_rate": overall_metrics.parse_valid_rate,
    },
    "pairing_checks": pairing_checks,
    "per_split": {
        split_name: {
            "binary_accuracy": benchmark_results_by_split[split_name].metrics.binary_accuracy,
            "narrative_accuracy": benchmark_results_by_split[split_name].metrics.narrative_accuracy,
            "parse_valid_rate": benchmark_results_by_split[split_name].metrics.parse_valid_rate,
        }
        for split_name in _LOADED_PARTITIONS
    },
}
dry_run_summary


In [ ]:
def _fmt(value: float) -> str:
    return f"{value:.6f}"


report_lines = [
    "# Kaggle Staging Dry Run Summary",
    "",
    "- Packaged frozen artifacts are the source of truth for this dry run.",
    "- Binary-only headline metric: Post-shift Probe Accuracy = " + _fmt(overall_metrics.post_shift_probe_accuracy),
    "- Narrative companion accuracy = " + _fmt(overall_metrics.narrative_accuracy),
    "- Combined parse-valid rate across Binary and Narrative = " + _fmt(overall_metrics.parse_valid_rate),
    "- Narrative does not change the headline score; it remains required same-episode robustness evidence.",
    "- This is a staging dry run over packaged frozen artifacts, not a live external inference result.",
    "",
    "## Per Split",
    "",
    "| Split | Binary accuracy | Narrative accuracy | Parse-valid rate | same_episode_order | same_probe_targets |",
    "| --- | ---: | ---: | ---: | --- | --- |",
]

for split_name in _LOADED_PARTITIONS:
    metrics = benchmark_results_by_split[split_name].metrics
    checks = pairing_checks[split_name]
    report_lines.append(
        f"| {split_name} | {_fmt(metrics.binary_accuracy)} | {_fmt(metrics.narrative_accuracy)} | {_fmt(metrics.parse_valid_rate)} | {checks['same_episode_order']} | {checks['same_probe_targets']} |"
    )

report_lines.extend(
    [
        "",
        "## Boundary Notes",
        "",
        "- Binary remains the sole leaderboard-primary path.",
        "- Narrative is included as required non-leaderboard robustness evidence on the same frozen episodes and targets.",
        "- Only the final four labels are scored; any Narrative reasoning is unscored.",
        "- Scoring and reporting stay within the current v1 protocol boundary.",
    ]
)

report_markdown = "\n".join(report_lines)
print(report_markdown)
report_markdown


## Staging Notes

- Binary is the only leaderboard-primary task.
- Narrative is required non-leaderboard robustness evidence on the same frozen episodes and probe targets as Binary; only the final four labels are scored.
- Binary-only Post-shift Probe Accuracy is the primary metric.
- Narrative does not change the headline score.
- The notebook stages the frozen repaired implementation from packaged artifacts; it does not supersede local validation.
